# InkNotes formula model training

This trains the small handwriting → LaTeX model for InkNotes, then tunes it on your own writing.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.

* It asks you to connect Google Drive. Everything is saved there, so if Colab disconnects you can just **Run all** again and it carries on where it stopped.
* It asks you to upload **inknotes-dave.jsonl** (skip that if you've put it in the `InkNotes-training` folder in your Drive).
* The whole run takes about 4½ hours. Keep this tab open.
* At the end it downloads **inkmath-model.zip**. Upload that file to Claude.

The model learns from Google's MathWriting data set (CC BY-NC-SA 4.0, fine for personal, non-commercial use).

In [ ]:
# Settings
TRAIN_HOURS = 3.0        # main training time (total, across restarts)
FOLD_MINUTES = 8         # each of the 2 honest-score fine-tunes
FINAL_MINUTES = 12       # final fine-tune on all your writing
MAX_FRAMES = 48000       # batch size (input points per batch); lower it if the GPU runs out of memory

In [ ]:
import os, subprocess, sys
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE. Runtime → Change runtime type → T4 GPU, then Run all again')
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/InkNotes-training'
os.makedirs(DRIVE, exist_ok=True)
!pip -q install onnx onnxscript onnxruntime

In [ ]:
%%writefile inkmath.py
"""InkMath: a small ink -> LaTeX recogniser for InkNotes.

Input is online ink (strokes of x, y points, y pointing down, any scale).  Output is canonical LaTeX tokens.
Architecture: per-point features -> 2 strided convs (4x shorter) -> pre-norm Transformer encoder -> CTC.

The preprocessing here is mirrored exactly in core/src/inkmath.rs (the on-device version); if you change
anything in `prepare` or `features`, change it there too, or the app will see different input than training did.
"""
import json, math, os, re, random
import numpy as np

# ---------------------------------------------------------------- preprocessing (mirrored in Rust)
COARSE = 0.03     # first resampling step, in units of the ink's normalised height
STEP = 0.06       # final resampling step
MAXP = 1600       # max points fed to the model (step grows for very long inks)


def _scale(strokes):
    pts = np.concatenate([s for s in strokes if len(s)], 0)
    x0, y0 = pts.min(0)
    x1, y1 = pts.max(0)
    h, w = y1 - y0, x1 - x0
    s = max(h, 0.15 * w, 1e-6)
    return x0, y0, s


def resample(strokes, step):
    """Uniform arclength resampling: each stroke is sampled at distances 0, step, 2*step, ... along its path, plus its
    last point if that is more than 0.3*step past the last sample.  Returns (m,3): x, y, stroke-start flag."""
    out = []
    for s in strokes:
        s = np.asarray(s, dtype=np.float64)
        if len(s) == 0:
            continue
        d = np.hypot(np.diff(s[:, 0]), np.diff(s[:, 1]))
        c = np.concatenate([[0.0], np.cumsum(d)])
        total = c[-1]
        k = int(math.floor(total / step + 1e-9))
        t = np.arange(k + 1) * step
        x = np.interp(t, c, s[:, 0])
        y = np.interp(t, c, s[:, 1])
        if total - t[-1] > 0.3 * step:
            x = np.append(x, s[-1, 0]); y = np.append(y, s[-1, 1])
        f = np.zeros(len(x)); f[0] = 1.0
        out.append(np.stack([x, y, f], 1))
    if not out:
        return np.zeros((0, 3), np.float32)
    return np.concatenate(out, 0).astype(np.float32)


def split3(p):
    """(m,3) x,y,start -> list of (n,2) strokes"""
    starts = list(np.nonzero(p[:, 2] > 0.5)[0]) + [len(p)]
    return [p[starts[i]:starts[i + 1], :2] for i in range(len(starts) - 1)]


def coarse(strokes):
    """Raw strokes (any units) -> normalised, coarsely resampled points (m,3). This is what the training cache stores."""
    strokes = [np.asarray(s, dtype=np.float64)[:, :2] for s in strokes if len(s)]
    x0, y0, s = _scale(strokes)
    strokes = [np.stack([(a[:, 0] - x0) / s, (a[:, 1] - y0) / s], 1) for a in strokes]
    return resample(strokes, COARSE)


def features(p):
    """Coarse points (m,3) -> model input (n,5): dx/step, dy/step, y, stroke start, stroke end."""
    strokes = split3(p)
    if not strokes:
        return np.zeros((1, 5), np.float32)
    x0, y0, s = _scale(strokes)                      # renormalise (augmentation may have changed the extent)
    strokes = [np.stack([(a[:, 0] - x0) / s, (a[:, 1] - y0) / s], 1) for a in strokes]
    length = sum(float(np.hypot(*np.diff(a, axis=0).T).sum()) if len(a) > 1 else 0.0 for a in strokes)
    step = STEP
    if length / step + len(strokes) > MAXP:
        step = (length + 1e-6) / (MAXP - len(strokes) - 2)
    q = resample(strokes, step)
    n = len(q)
    f = np.zeros((n, 5), np.float32)
    f[1:, 0] = (q[1:, 0] - q[:-1, 0]) / step
    f[1:, 1] = (q[1:, 1] - q[:-1, 1]) / step
    f[:, 2] = q[:, 1]
    f[:, 3] = q[:, 2]
    f[:-1, 4] = q[1:, 2]
    f[-1, 4] = 1.0
    np.clip(f[:, :2], -8.0, 8.0, out=f[:, :2])
    return f


def augment(p, rng):
    """Random writer-style distortion of coarse points (training only)."""
    q = p.copy()
    x, y = q[:, 0].astype(np.float64), q[:, 1].astype(np.float64)
    sx = math.exp(rng.uniform(-0.25, 0.25))
    sh = rng.uniform(-0.35, 0.25)                    # slant
    rot = math.radians(rng.uniform(-6, 6))
    x = x * sx + sh * (y - 0.5)
    c, s = math.cos(rot), math.sin(rot)
    x, y = c * x - s * y, s * x + c * y
    # gentle per-stroke wobble
    starts = list(np.nonzero(q[:, 2] > 0.5)[0]) + [len(q)]
    for i in range(len(starts) - 1):
        a, b = starts[i], starts[i + 1]
        x[a:b] += rng.normal(0, 0.015)
        y[a:b] += rng.normal(0, 0.015)
    q[:, 0], q[:, 1] = x, y
    return q


# ---------------------------------------------------------------- LaTeX tokens (canonicalisation mirrored in Rust)
TOK = re.compile(r"\\[A-Za-z]+|\\.|[^\s]")
DROP = {"\\left", "\\right", "\\,", "\\;", "\\!", "\\ ", "\\quad", "\\qquad", "\\displaystyle", "\\limits", "\\nolimits", "\\:", "\\big", "\\Big", "\\bigg", "\\Bigg"}
SYN = {"\\le": "\\leq", "\\ge": "\\geq", "\\ne": "\\neq", "\\dfrac": "\\frac", "\\tfrac": "\\frac", "\\lbrace": "\\{", "\\rbrace": "\\}",
       "\\to": "\\rightarrow", "\\gets": "\\leftarrow", "\\varepsilon": "\\epsilon", "\\vartheta": "\\theta", "\\varphi": "\\phi",
       "\\lvert": "|", "\\rvert": "|", "\\vert": "|", "\\ast": "*", "−": "-", "×": "\\times", "·": "\\cdot"}


def canon(latex):
    """LaTeX string -> canonical token list (scripts always braced, spacing and \\left/\\right dropped)."""
    t = [SYN.get(x, x) for x in TOK.findall(latex) if x not in DROP]
    out = []
    i = 0
    while i < len(t):
        out.append(t[i])
        if t[i] in ("^", "_") and i + 1 < len(t) and t[i + 1] != "{":
            out += ["{", t[i + 1], "}"]
            i += 2
            continue
        i += 1
    return out


class Vocab:
    def __init__(self, toks):
        self.toks = list(toks)            # index 0 is the CTC blank
        self.idx = {t: i for i, t in enumerate(self.toks)}

    @staticmethod
    def build(label_iter, min_count=20):
        from collections import Counter
        c = Counter()
        for lab in label_iter:
            c.update(canon(lab))
        return Vocab(["<blank>"] + sorted(t for t, n in c.items() if n >= min_count))

    def encode(self, toks):
        ids = [self.idx.get(t) for t in toks]
        return None if any(i is None or i == 0 for i in ids) else ids

    def save(self, path):
        json.dump({"tokens": self.toks, "coarse": COARSE, "step": STEP, "maxp": MAXP}, open(path, "w"), ensure_ascii=False, indent=0)

    @staticmethod
    def load(path):
        return Vocab(json.load(open(path))["tokens"])


def join(toks):
    """tokens -> LaTeX text (same as join_latex in Rust)"""
    s = ""
    for x in toks:
        m = re.search(r"\\[A-Za-z]+$", s)
        if m and x[:1].isascii() and x[:1].isalpha():
            s += " "
        s += x
    return s


def ctc_greedy(logp, vocab):
    """logp: (T, V) log-probs -> (tokens, confidence)"""
    best = logp.argmax(-1)
    pmax = np.exp(logp.max(-1))
    toks, conf, prev = [], 1.0, 0
    for i, k in enumerate(best):
        k = int(k)
        if k != prev and k != 0:
            toks.append(vocab.toks[k])
            conf = min(conf, float(pmax[i]))
        prev = k
    return toks, conf


# ---------------------------------------------------------------- model
def make_model(nvocab, d=256, layers=8, heads=4, ff=1024, drop=0.1):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class Block(nn.Module):
        def __init__(s):
            super().__init__()
            s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
            s.qkv, s.o = nn.Linear(d, 3 * d), nn.Linear(d, d)
            s.f1, s.f2 = nn.Linear(d, ff), nn.Linear(ff, d)
            s.drop = nn.Dropout(drop)

        def forward(s, x, bias):
            B, T, _ = x.shape
            q, k, v = s.qkv(s.n1(x)).view(B, T, 3, heads, d // heads).permute(2, 0, 3, 1, 4)
            a = torch.matmul(q, k.transpose(-1, -2)) * (1.0 / math.sqrt(d // heads))
            if bias is not None:
                a = a + bias
            a = torch.softmax(a, -1)
            y = torch.matmul(s.drop(a), v).transpose(1, 2).reshape(B, T, d)
            x = x + s.drop(s.o(y))
            return x + s.drop(s.f2(s.drop(F.gelu(s.f1(s.n2(x))))))

    class Net(nn.Module):
        def __init__(s):
            super().__init__()
            s.cfg = dict(nvocab=nvocab, d=d, layers=layers, heads=heads, ff=ff)
            s.inp = nn.Linear(5, d)
            s.c1 = nn.Conv1d(d, d, 5, stride=2, padding=2)
            s.c2 = nn.Conv1d(d, d, 5, stride=2, padding=2)
            pe = torch.zeros(4096, d)
            pos = torch.arange(4096).float()[:, None]
            div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
            pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
            s.register_buffer("pe", pe, persistent=False)
            s.blocks = nn.ModuleList(Block() for _ in range(layers))
            s.norm = nn.LayerNorm(d)
            s.out = nn.Linear(d, nvocab)

        @staticmethod
        def out_len(n):
            return (((n + 1) // 2) + 1) // 2

        def forward(s, x, lengths=None):
            """x: (B, T, 5) -> log-probs (B, T', V)"""
            h = F.gelu(s.inp(x)).transpose(1, 2)
            h = F.gelu(s.c2(F.gelu(s.c1(h)))).transpose(1, 2)
            T = h.shape[1]
            h = h + s.pe[:T]
            bias = None
            if lengths is not None:
                ol = s.out_len(lengths)
                pad = torch.arange(T, device=x.device)[None, :] >= ol[:, None]
                bias = pad[:, None, None, :].float() * -1e4
            for b in s.blocks:
                h = b(h, bias)
            return torch.log_softmax(s.out(s.norm(h)), -1)

    return Net()


def export_onnx(model, path):
    import torch
    model = model.eval().float().cpu()
    x = torch.randn(1, 300, 5)
    torch.onnx.export(model, (x,), path, input_names=["x"], output_names=["logp"],
                      dynamic_axes={"x": {1: "T"}, "logp": {1: "U"}}, opset_version=17, dynamo=False)


# ---------------------------------------------------------------- data
def read_inkml(path):
    return read_inkml_text(open(path, encoding="utf-8").read())


def read_inkml_text(text):
    from xml.etree import ElementTree
    root = ElementTree.fromstring(text)
    strokes, ann = [], {}
    for el in root:
        tag = el.tag.split("}")[-1]
        if tag == "annotation":
            ann[el.attrib.get("type")] = el.text or ""
        elif tag == "trace":
            pts = [p.split() for p in el.text.strip().split(",")]
            strokes.append(np.array([[float(a[0]), float(a[1])] for a in pts if len(a) >= 2]))
    return strokes, ann


def inknotes_strokes(rec_strokes):
    """InkNotes export stroke objects -> list of (n,2) arrays"""
    return [np.array([[p[0], p[1]] for p in s["p"]], dtype=np.float64) for s in rec_strokes if s.get("p")]


class Cache:
    """All training inks as coarse points in one float16 array + offsets (memory-mappable)."""

    def __init__(self, pts, off, labels, src, base=None):
        self.pts, self.off, self.labels, self.src, self.base = pts, off, labels, src, base

    def __getstate__(self):
        if self.base:       # worker processes reopen the files rather than receive a copy of the data
            return {"base": self.base}
        return self.__dict__

    def __setstate__(self, st):
        if list(st) == ["base"]:
            self.__dict__.update(Cache.load(st["base"]).__dict__)
        else:
            self.__dict__.update(st)

    def __len__(self):
        return len(self.labels)

    def get(self, i):
        return self.pts[self.off[i]:self.off[i + 1]].astype(np.float32)

    def save(self, base):
        np.save(base + "-pts.npy", self.pts)
        np.save(base + "-off.npy", self.off)
        json.dump({"labels": self.labels, "src": self.src}, open(base + "-meta.json", "w"))

    @staticmethod
    def load(base, mmap=True):
        m = json.load(open(base + "-meta.json"))
        return Cache(np.load(base + "-pts.npy", mmap_mode="r" if mmap else None), np.load(base + "-off.npy"), m["labels"], m["src"], base if mmap else None)

    @staticmethod
    def build(items):
        """items: iterable of (coarse points (m,3), label ids list, source tag)"""
        chunks, off, labels, src = [], [0], [], []
        for p, ids, tag in items:
            chunks.append(p.astype(np.float16))
            off.append(off[-1] + len(p))
            labels.append(ids)
            src.append(tag)
        pts = np.concatenate(chunks, 0) if chunks else np.zeros((0, 3), np.float16)
        return Cache(pts, np.array(off, np.int64), labels, src)


def batches(cache, idx, max_frames, rng, aug=True):
    """Length-bucketed batches of (features list, label list)."""
    idx = list(idx)
    lens = np.diff(cache.off)
    idx.sort(key=lambda i: lens[i] + rng.random() * 40)
    groups, cur, cur_max = [], [], 0
    for i in idx:
        L = int(lens[i] * COARSE / STEP) + 8
        m = max(cur_max, L)
        if cur and m * (len(cur) + 1) > max_frames:
            groups.append(cur)
            cur, m = [], L
        cur.append(i)
        cur_max = m
    if cur:
        groups.append(cur)
    rng.shuffle(groups)
    for g in groups:
        feats = []
        for i in g:
            p = cache.get(i)
            if aug:
                p = augment(p, rng)
            feats.append(features(p))
        yield feats, [cache.labels[i] for i in g]


def collate(feats, labels, device):
    import torch
    T = max(len(f) for f in feats)
    x = np.zeros((len(feats), T, 5), np.float32)
    for k, f in enumerate(feats):
        x[k, :len(f)] = f
    lengths = torch.tensor([len(f) for f in feats])
    tgt = torch.tensor([t for l in labels for t in l], dtype=torch.long)
    tl = torch.tensor([len(l) for l in labels])
    return torch.from_numpy(x).to(device), lengths.to(device), tgt, tl


def decode_batch(logp, lengths, model, vocab):
    out = []
    ol = model.out_len(lengths).cpu().numpy()
    lp = logp.float().cpu().numpy()
    for k in range(lp.shape[0]):
        out.append(ctc_greedy(lp[k, :ol[k]], vocab))
    return out

In [ ]:
%%writefile run.py
"""Training pipeline for the InkMath model.  Used by the Colab notebook; can also be run as a script.

Steps: prepare (MathWriting InkML -> cache), train (CTC, time-budgeted, resumable), evaluate on the
InkNotes recordings, fine-tune (2-fold by prompt for an honest score, then on everything), export ONNX.
"""
import glob, json, math, os, random, time
from multiprocessing import Pool
import numpy as np
import torch
import inkmath as im


# ---------------------------------------------------------------- data preparation
def _load_one(path):
    try:
        strokes, ann = im.read_inkml(path)
        lab = ann.get("normalizedLabel") or ann.get("label") or ""
        if not strokes or not lab:
            return None
        return im.coarse(strokes).astype(np.float16), lab
    except Exception:
        return None


SPLITS = ["train", "synthetic", "symbols", "valid", "test"]


def prepare(mw_dir, out_dir, workers=None, limit=None, log=print):
    """Read every InkML file of an unpacked MathWriting folder once; writes train/valid/test caches and vocab.json."""
    files = {sp: sorted(glob.glob(os.path.join(mw_dir, sp, "*.inkml"))) for sp in SPLITS}
    raw = {}
    with Pool(workers or os.cpu_count()) as pool:
        for sp, f in files.items():
            t = time.time()
            raw[sp] = [r for r in pool.imap(_load_one, f[:limit] if limit else f, chunksize=256) if r is not None]
            log(f"  read {sp}: {len(raw[sp])} inks in {time.time() - t:.0f}s")
    return _finish(raw, out_dir, small=bool(limit), log=log)


def _parse_member(item):
    name, data = item
    try:
        strokes, ann = im.read_inkml_text(data.decode("utf-8"))
        lab = ann.get("normalizedLabel") or ann.get("label") or ""
        if not strokes or not lab:
            return None
        return name.replace("\\", "/").split("/")[-2], im.coarse(strokes).astype(np.float16), lab
    except Exception:
        return None


def prepare_tgz(tgz, out_dir, workers=None, small=False, log=print):
    """Same as prepare(), reading straight from mathwriting-2024.tgz (no unpacking of 650k small files)."""
    import tarfile
    raw = {sp: [] for sp in SPLITS}
    t, n = time.time(), 0
    import multiprocessing as mp
    ctx = mp.get_context("spawn" if os.name == "nt" or os.environ.get("INKMATH_SPAWN") else None)
    with ctx.Pool(workers or max(1, (os.cpu_count() or 2) - 1)) as pool, tarfile.open(tgz, "r|gz") as tf:
        chunk = []
        def flush():
            for r in pool.map(_parse_member, chunk, chunksize=64):
                if r is not None and r[0] in raw:
                    raw[r[0]].append(r[1:])
            chunk.clear()
        for m in tf:
            if m.isfile() and m.name.endswith(".inkml"):
                chunk.append((m.name, tf.extractfile(m).read()))
                n += 1
                if len(chunk) >= 20000:
                    flush()
                    log(f"  {n} inks read ({time.time() - t:.0f}s)")
        flush()
    log("  " + ", ".join(f"{sp}: {len(v)}" for sp, v in raw.items()) + f"  ({time.time() - t:.0f}s)")
    return _finish(raw, out_dir, small=small, log=log)


def _finish(raw, out_dir, small=False, log=print):
    os.makedirs(out_dir, exist_ok=True)
    vocab = im.Vocab.build((r[1] for sp in ["train", "synthetic", "symbols"] for r in raw[sp]), min_count=1 if small else 20)
    vocab.save(os.path.join(out_dir, "vocab.json"))
    log(f"vocab: {len(vocab.toks)} tokens")
    for name, sps in [("train", ["train", "synthetic", "symbols"]), ("valid", ["valid"]), ("test", ["test"])]:
        items, dropped = [], 0
        for sp in sps:
            for r in raw[sp]:
                p, lab = r[0], r[1]
                ids = vocab.encode(im.canon(lab))
                if ids is None or len(p) < 2:
                    dropped += 1
                    continue
                items.append((p, ids, sp))
        c = im.Cache.build(items)
        c.save(os.path.join(out_dir, name))
        log(f"{name}: {len(c)} inks ({dropped} dropped)")
    return vocab


def load_inknotes(path, vocab):
    """InkNotes fine-tune JSONL -> list of dict(p=coarse points, ids, prompt, kind)"""
    out = []
    for line in open(path, encoding="utf-8"):
        r = json.loads(line)
        strokes = [np.array(s, dtype=np.float64).reshape(-1, 2) for s in r["strokes"] if len(s)]
        if not strokes:
            continue
        toks = im.canon(r["latex"])
        ids = vocab.encode(toks)
        if ids is None:
            print("  skipped (token not in vocabulary):", r["latex"])
            continue
        out.append(dict(p=im.coarse(strokes), ids=ids, toks=toks, prompt=r["prompt"], kind=r["kind"], latex=r["latex"], unsure=bool(r.get("unsure"))))
    return out


# ---------------------------------------------------------------- training
def _loader(stream, workers):
    ctx = "spawn" if (os.name == "nt" or os.environ.get("INKMATH_SPAWN")) and workers else None
    return torch.utils.data.DataLoader(stream, batch_size=None, num_workers=workers, prefetch_factor=4 if workers else None,
                                       multiprocessing_context=ctx, persistent_workers=False)


def device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def save_ckpt(path, model, opt, sched_step, extra):
    tmp = path + ".tmp"
    torch.save({"model": model.state_dict(), "opt": opt.state_dict() if opt else None, "step": sched_step, "cfg": model.cfg, **extra}, tmp)
    os.replace(tmp, path)


def load_model(path, dev=None):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = ck["cfg"]
    m = im.make_model(cfg["nvocab"], d=cfg["d"], layers=cfg["layers"], heads=cfg["heads"], ff=cfg["ff"])
    m.load_state_dict(ck["model"])
    return m.to(dev or device()), ck


class Mixed:
    """A cache-like view over MathWriting inks plus (oversampled) InkNotes inks."""

    def __init__(self, base, extra, reps):
        self.base, self.extra, self.reps = base, extra, reps
        self.nb = len(base) if base is not None else 0

    def items(self):
        return list(range(self.nb)) + [self.nb + i for i in range(len(self.extra)) for _ in range(self.reps)]

    def length(self, i):
        return (self.base.off[i + 1] - self.base.off[i]) if i < self.nb else len(self.extra[i - self.nb]["p"])

    def get(self, i):
        return (self.base.get(i), self.base.labels[i]) if i < self.nb else (self.extra[i - self.nb]["p"], self.extra[i - self.nb]["ids"])


def mixed_batches(ds, idx, max_frames, rng, aug=True):
    idx = list(idx)
    idx.sort(key=lambda i: ds.length(i) + rng.random() * 40)
    groups, cur, cur_max = [], [], 0
    for i in idx:
        L = int(ds.length(i) * im.COARSE / im.STEP) + 8
        m = max(cur_max, L)
        if cur and m * (len(cur) + 1) > max_frames:
            groups.append(cur)
            cur, m = [], L
        cur.append(i)
        cur_max = m
    if cur:
        groups.append(cur)
    rng.shuffle(groups)
    for g in groups:
        feats, labs = [], []
        for i in g:
            p, ids = ds.get(i)
            feats.append(im.features(im.augment(p, rng) if aug else p))
            labs.append(ids)
        yield feats, labs


class _Stream(torch.utils.data.IterableDataset):
    def __init__(self, ds, max_frames, seed, aug=True, sub=None):
        self.ds, self.max_frames, self.seed, self.aug, self.sub = ds, max_frames, seed, aug, sub

    def __iter__(self):
        info = torch.utils.data.get_worker_info()
        wid, nw = (info.id, info.num_workers) if info else (0, 1)
        epoch = 0
        while True:
            rng = np.random.default_rng(self.seed * 1000 + epoch * 17 + wid)
            idx = self.ds.items() if self.sub is None else self.sub
            idx = [i for k, i in enumerate(idx) if k % nw == wid]
            for b in mixed_batches(self.ds, idx, self.max_frames, rng, self.aug):
                yield b
            epoch += 1


def train(ds, vocab, ckpt, hours, max_frames=48000, peak_lr=1e-3, warm=2000, workers=2,
          d=256, layers=8, heads=4, ff=1024, eval_fn=None, eval_every=2000, save_every_min=15, seed=0, log=print):
    """Trains for `hours` of training time in total (across restarts: progress is kept in the checkpoint).
    The learning rate warms up over `warm` steps, then follows a cosine over the time budget."""
    dev = device()
    elapsed = 0.0
    if os.path.exists(ckpt):
        model, ck = load_model(ckpt, dev)
        opt = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.01)
        if ck.get("opt"):
            opt.load_state_dict(ck["opt"])
        start, elapsed = ck["step"], ck.get("elapsed", 0.0)
        log(f"resuming from step {start} ({elapsed / 3600:.2f} h done)")
    else:
        model = im.make_model(len(vocab.toks), d=d, layers=layers, heads=heads, ff=ff).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.01)
        start = 0
    log(f"model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters on {dev}")
    budget = hours * 3600
    if elapsed >= budget:
        log("time budget already used; nothing to do")
        model.eval()
        return model
    loader = _loader(_Stream(ds, max_frames, seed + start), workers)
    scaler = torch.amp.GradScaler(enabled=dev.type == "cuda")
    ctc = torch.nn.CTCLoss(blank=0, zero_infinity=True)
    t_last = time.time()
    t_save = time.time() + save_every_min * 60
    step, run_loss, t0 = start, None, time.time()
    model.train()
    for feats, labs in loader:
        x, lengths, tgt, tl = im.collate(feats, labs, dev)
        frac = min(1.0, elapsed / budget)
        lr = peak_lr * min(1.0, (step + 1) / warm) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * frac)))
        for g in opt.param_groups:
            g["lr"] = lr
        with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
            logp = model(x, lengths)
        ol = model.out_len(lengths)
        loss = ctc(logp.float().transpose(0, 1), tgt.to(dev), ol, tl.to(dev))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        step += 1
        lv = float(loss.detach())
        run_loss = lv if run_loss is None else 0.98 * run_loss + 0.02 * lv
        now = time.time()
        elapsed += now - t_last
        t_last = now
        if step % 200 == 0:
            log(f"step {step}  loss {run_loss:.3f}  lr {lr:.2e}  {(now - t0) / 200:.2f}s/step  {elapsed / 3600:.2f}/{hours} h")
            t0 = now
        if eval_fn and step % eval_every == 0:
            model.eval()
            log("  " + eval_fn(model))
            model.train()
            t_last = time.time()
        done = elapsed >= budget
        if time.time() > t_save or done:
            save_ckpt(ckpt, model, opt, step, {"vocab": vocab.toks, "elapsed": elapsed})
            t_save = time.time() + save_every_min * 60
            t_last = time.time()
        if done:
            break
    model.eval()
    return model


# ---------------------------------------------------------------- evaluation
def edit(a, b):
    d = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(b) + 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
    return d[len(b)]


@torch.no_grad()
def recognise(model, vocab, points_list, max_frames=20000):
    dev = next(model.parameters()).device
    feats = [im.features(p) for p in points_list]
    out = [None] * len(feats)
    order = sorted(range(len(feats)), key=lambda i: len(feats[i]))
    k = 0
    while k < len(order):
        g = [order[k]]
        while k + len(g) < len(order) and len(feats[order[k + len(g)]]) * (len(g) + 1) <= max_frames:
            g.append(order[k + len(g)])
        x, lengths, _, _ = im.collate([feats[i] for i in g], [[1] for _ in g], dev)
        res = im.decode_batch(model(x, lengths), lengths, model, vocab)
        for i, r in zip(g, res):
            out[i] = r
        k += len(g)
    return out


def score(model, vocab, items):
    """items: list of (coarse points, reference token list). Returns (exact-match rate, token error rate)."""
    res = recognise(model, vocab, [p for p, _ in items])
    ok = sum(r[0] == ref for r, (_, ref) in zip(res, items))
    err = sum(edit(r[0], ref) for r, (_, ref) in zip(res, items))
    n = sum(len(ref) for _, ref in items)
    return ok / max(1, len(items)), err / max(1, n), res


def valid_eval_fn(valid, vocab, n=1500, seed=0):
    rng = random.Random(seed)
    sel = rng.sample(range(len(valid)), min(n, len(valid)))
    items = [(valid.get(i), [vocab.toks[t] for t in valid.labels[i]]) for i in sel]

    def f(model):
        em, ter, _ = score(model, vocab, items)
        return f"valid: exact {em:.1%}  token errors {ter:.1%}"
    return f


def inknotes_report(model, vocab, dave, title, kind="formula"):
    forms = [d for d in dave if d["kind"] == kind and not d.get("unsure")]
    em, ter, res = score(model, vocab, [(d["p"], d["toks"]) for d in forms])
    what = "practice formulas" if kind == "formula" else "homework lines"
    lines = [f"{title}: {em:.1%} of {len(forms)} {what} exactly right, token errors {ter:.1%}"]
    for d, r in zip(forms, res):
        lines.append(f"  {'✓' if r[0] == d['toks'] else '✗'} {d['latex']:<40} got {im.join(r[0])}   conf {r[1]:.2f}")
    return em, "\n".join(lines)


# ---------------------------------------------------------------- fine-tuning on InkNotes recordings
def finetune(base_ckpt, mw_train, dave_items, vocab, out_ckpt, minutes, reps=40, peak_lr=2e-4, max_frames=48000, workers=2, log=print):
    ds = Mixed(mw_train, dave_items, reps)
    if mw_train is not None:
        # keep a slice of MathWriting in the mix so the model doesn't forget general math
        rng = random.Random(1)
        n_dave = len(dave_items) * reps
        base_ids = rng.sample(range(len(mw_train)), min(len(mw_train), 2 * n_dave))
        sub = base_ids + [len(mw_train) + i for i in range(len(dave_items)) for _ in range(reps)]
    else:
        sub = None
    if os.path.exists(out_ckpt):
        os.remove(out_ckpt)
    dev = device()
    model, _ = load_model(base_ckpt, dev)
    opt = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.01)
    ctc = torch.nn.CTCLoss(blank=0, zero_infinity=True)
    scaler = torch.amp.GradScaler(enabled=dev.type == "cuda")
    loader = _loader(_Stream(ds, max_frames, 7, sub=sub), workers)
    t_end = time.time() + minutes * 60
    step = 0
    total = None
    model.train()
    for feats, labs in loader:
        x, lengths, tgt, tl = im.collate(feats, labs, dev)
        frac = min(1.0, (time.time() - (t_end - minutes * 60)) / (minutes * 60))
        for g in opt.param_groups:
            g["lr"] = peak_lr * (min(1.0, step / 100) * (0.5 + 0.5 * math.cos(math.pi * frac)))
        with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
            logp = model(x, lengths)
        loss = ctc(logp.float().transpose(0, 1), tgt.to(dev), model.out_len(lengths), tl.to(dev))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        step += 1
        if step % 100 == 0:
            log(f"  fine-tune step {step} loss {float(loss.detach()):.3f}")
        if time.time() > t_end:
            break
    model.eval()
    save_ckpt(out_ckpt, model, None, step, {"vocab": vocab.toks})
    return model


def prompt_folds(dave, k=2, seed=0):
    prompts = sorted(set(d["prompt"] for d in dave))
    random.Random(seed).shuffle(prompts)
    return [set(prompts[i::k]) for i in range(k)]


# ---------------------------------------------------------------- export
def export(ckpt, vocab_path, out_dir, report):
    os.makedirs(out_dir, exist_ok=True)
    model, _ = load_model(ckpt, torch.device("cpu"))
    im.export_onnx(model, os.path.join(out_dir, "inkmath.onnx"))
    import shutil
    shutil.copy(vocab_path, os.path.join(out_dir, "vocab.json"))
    open(os.path.join(out_dir, "report.txt"), "w", encoding="utf-8").write(report)
    # parity fixture: lets the app build check its own preprocessing + runner against PyTorch
    return out_dir

In [ ]:
import importlib, shutil, glob, json
import inkmath as im, run
importlib.reload(im); importlib.reload(run)
DAVE = os.path.join(DRIVE, 'inknotes-dave.jsonl')
if not os.path.exists(DAVE):
    from google.colab import files
    print('Upload inknotes-dave.jsonl')
    up = files.upload()
    name = next(iter(up))
    open(DAVE, 'wb').write(up[name])
HOMEWORK = os.path.join(DRIVE, 'inknotes-homework.jsonl')
print('your recordings:', sum(1 for _ in open(DAVE)), 'inks;', 'homework lines:', sum(1 for _ in open(HOMEWORK)) if os.path.exists(HOMEWORK) else 'none (optional: put inknotes-homework.jsonl in the Drive folder)')

In [ ]:
# MathWriting -> cache (done once; kept in Drive)
CACHE = os.path.join(DRIVE, 'cache')
if not os.path.exists(os.path.join(CACHE, 'train-meta.json')):
    if not os.path.exists('/content/mathwriting-2024'):
        print('downloading MathWriting (2.9 GB)...')
        subprocess.run(['wget', '-q', '-nc', 'https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz', '-O', '/content/mw.tgz'], check=True)
        print('unpacking...')
        subprocess.run(['tar', '-xzf', '/content/mw.tgz', '-C', '/content'], check=True)
        os.remove('/content/mw.tgz')
    # in a separate process so the memory it uses is given back afterwards
    subprocess.run([sys.executable, '-c', "import run; run.prepare('/content/mathwriting-2024', '/content/cache')"], check=True)
    shutil.copytree('/content/cache', CACHE, dirs_exist_ok=True)
elif not os.path.exists('/content/cache/train-meta.json'):
    shutil.copytree(CACHE, '/content/cache', dirs_exist_ok=True)
vocab = im.Vocab.load('/content/cache/vocab.json')
train_c = im.Cache.load('/content/cache/train')
valid_c = im.Cache.load('/content/cache/valid')
dave = run.load_inknotes(DAVE, vocab)
if os.path.exists(HOMEWORK):
    dave += run.load_inknotes(HOMEWORK, vocab)
print(len(vocab.toks), 'tokens;', len(train_c), 'training inks;', len(dave), 'of your inks usable')

In [ ]:
# Main training (resumes from Drive if interrupted)
log = open(os.path.join(DRIVE, 'log.txt'), 'a')
def say(s):
    print(s); log.write(s + '\n'); log.flush()
BASE = os.path.join(DRIVE, 'base.pt')
model = run.train(run.Mixed(train_c, [], 0), vocab, BASE, hours=TRAIN_HOURS, max_frames=MAX_FRAMES,
                  eval_fn=run.valid_eval_fn(valid_c, vocab), eval_every=3000, log=say)

In [ ]:
# How well does it read your writing before seeing any of it?
model, _ = run.load_model(BASE)
em0, rep0 = run.inknotes_report(model, vocab, dave, 'Before tuning on your writing')
say(rep0.split('\n')[0])
if any(d['kind'] == 'homework' for d in dave):
    _, rep0h = run.inknotes_report(model, vocab, dave, 'Before tuning on your writing', kind='homework')
    say(rep0h.split('\n')[0]); rep0 += '\n\n' + rep0h

In [ ]:
# Honest score after tuning: tune on half of your prompts, test on the other half (and swap)
folds = run.prompt_folds(dave)
rep_f, right, total, hr, ht = [], 0, 0, 0, 0
for k, held in enumerate(folds):
    tr = [d for d in dave if d['prompt'] not in held]
    te = [d for d in dave if d['prompt'] in held and d['kind'] == 'formula']
    fm = run.finetune(BASE, train_c, tr, vocab, f'/content/fold{k}.pt', minutes=FOLD_MINUTES, max_frames=MAX_FRAMES, log=say)
    em, rep = run.inknotes_report(fm, vocab, te, f'Fold {k + 1} (formulas it never saw)')
    right += round(em * len(te)); total += len(te)
    say(rep.split('\n')[0]); rep_f.append(rep)
    teh = [d for d in dave if d['prompt'] in held and d['kind'] == 'homework']
    if teh:
        emh, reph = run.inknotes_report(fm, vocab, teh, f'Fold {k + 1} (homework notes it never saw)', kind='homework')
        hr += round(emh * len([d for d in teh if not d['unsure']])); ht += len([d for d in teh if not d['unsure']])
        say(reph.split('\n')[0]); rep_f.append(reph)
say(f'Tuned, on formulas it never saw: {right}/{total}')
if ht: say(f'Tuned, on homework notes it never saw: {hr}/{ht}')

In [ ]:
# Final model: tuned on all of your writing; export and download
FINAL = os.path.join(DRIVE, 'final.pt')
fm = run.finetune(BASE, train_c, dave, vocab, FINAL, minutes=FINAL_MINUTES, max_frames=MAX_FRAMES, log=say)
em1, rep1 = run.inknotes_report(fm, vocab, dave, 'Final model on all your formulas (it has seen these)')
report = '\n\n'.join([rep0] + rep_f + [rep1, open(os.path.join(DRIVE, 'log.txt')).read()[-20000:]])
out = '/content/inkmath-model'
run.export(FINAL, '/content/cache/vocab.json', out, report)
m0, _ = run.load_model(BASE, run.torch.device('cpu')); im.export_onnx(m0, os.path.join(out, 'inkmath-base.onnx'))
shutil.make_archive('/content/inkmath-model', 'zip', out)
shutil.copy('/content/inkmath-model.zip', DRIVE)
from google.colab import files
files.download('/content/inkmath-model.zip')
print('Done. Upload inkmath-model.zip to Claude (a copy is also in your Drive, InkNotes-training folder).')